# Italia

Quadro nazionale e analisi avanzata del bilancio italiano.
Usa i dati in `data/export/bilancio-pubblico/sections/italia.json` e produce grafici su più livelli: KPI, entrate, spesa e distribuzioni.


## Scaricamento dati

Esegui questa cella prima degli import di `utils_bilancio`. La cella usa solo Python standard per trovare il repository e, se serve, lancia `scripts/run_sections.py` per creare o aggiornare il JSON della sezione.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

SECTION_ID = "italia"
REFRESH = False
FORCE_DOWNLOAD = False
repo_root = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "scripts" / "utils_bilancio").is_dir():
        repo_root = candidate
        break

if repo_root is None:
    raise ModuleNotFoundError("Impossibile trovare il repository Bilancio_pubblico.")

section_file = repo_root / "data" / "export" / "bilancio-pubblico" / "sections" / f"{SECTION_ID}.json"
command = [sys.executable, str(repo_root / "scripts" / "run_sections.py"), "--sections", SECTION_ID]
if REFRESH or FORCE_DOWNLOAD:
    command.append("--refresh")

env = os.environ.copy()
if SECTION_ID == "regioni" and SIOPE_YEARS:
    env["BILANCIO_PUBBLICO_SIOPE_YEARS"] = SIOPE_YEARS

if FORCE_DOWNLOAD or REFRESH or not section_file.exists():
    print("Eseguo:", " ".join(command))
    subprocess.run(command, cwd=repo_root, check=True, env=env)
else:
    print(f"Uso cache: {section_file}")


## Import e caricamento

Dopo che i file sono presenti, questa cella importa gli helper del repo e carica `section`, `status`, `frame` e l'eventuale `source_payload`.


In [ ]:
from pathlib import Path
import sys

SECTION_ID = "italia"

if "repo_root" not in globals() or repo_root is None:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "scripts" / "utils_bilancio").is_dir():
            repo_root = candidate
            break
    else:
        raise ModuleNotFoundError("Impossibile trovare il repository Bilancio_pubblico.")

scripts_dir = repo_root / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

from utils_bilancio.notebook.utils import setup_notebook_section

notebook_state = setup_notebook_section(
    section_id=SECTION_ID,
    refresh=False,
    force_download=False,
    include_source_payload=SECTION_ID == "italia",
)

section = notebook_state["section"]
status = notebook_state["status"]
frame = notebook_state["frame"]
source_payload = notebook_state.get("source_payload")


## Parametri principali

Nella cella **Scaricamento dati** puoi modificare:
- `REFRESH = False/True`: aggiorna le fonti quando la sezione viene rigenerata.
- `FORCE_DOWNLOAD = False/True`: forza una nuova materializzazione anche se il JSON esiste gia'.

Nelle celle di analisi puoi regolare:
- `TOP_N`: intero, di solito tra `3` e `100`.
- `VALUE_PREFERENCE`: lista ordinata di colonne, per esempio `["latest_percent_pil", "latest_value_mld", "value", "value_mld"]`.

Esegui sempre **Elenco opzioni disponibili** prima di cambiare parametri: stampa le colonne e gli anni realmente presenti nella tua run.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 170)
plt.style.use("ggplot")



In [ ]:
def rows_to_df(rows):
    return frame(rows or [])


def pick_metric(record, options):
    for key in options:
        if key in record and pd.notna(record[key]):
            return key
    return None


def latest_year_from_df(df, year_column="year"):
    if df.empty or year_column not in df.columns:
        return None
    years = pd.to_numeric(df[year_column], errors="coerce")
    if years.dropna().empty:
        return None
    return int(years.max())


def top_bars(df, value_column, title, top=10, label_column="label"):
    if df.empty or value_column not in df.columns:
        print(f"Nessun dato per {title}")
        return
    subset = df.dropna(subset=[value_column]).copy()
    if subset.empty:
        print(f"Nessun dato per {title}")
        return
    selected = subset.sort_values(value_column, ascending=False).head(top).sort_values(value_column)
    selected.plot(x=label_column, y=value_column, kind="barh", figsize=(12, max(3.5, top * 0.35)), legend=False)
    plt.title(title)
    plt.tight_layout()
    plt.show()


def explode_series(category_rows, metric_field):
    output = []
    for category in category_rows or []:
        if not isinstance(category, dict):
            continue
        label = category.get("label") or category.get("categoria")
        code = category.get("code")
        group = category.get("group")
        for point in category.get("series") or []:
            if not isinstance(point, dict):
                continue
            value = point.get(metric_field)
            if value is None:
                continue
            output.append(
                {
                    "code": code,
                    "label": label,
                    "group": group,
                    "year": point.get("year") if point.get("year") is not None else point.get("anno"),
                    "value": value,
                    "percent_pil": point.get("percent_pil") or point.get("value_pil") or point.get("pil"),
                    "value_mld": point.get("value_mld") or point.get("mld"),
                }
            )
    return rows_to_df(output)


def plot_series(df, value_column, title, top=8, year_column="year", id_column="code", label_column="label"):
    source = df.dropna(subset=[value_column, year_column]).copy()
    if source.empty:
        print(f"Serie mancanti per {title}")
        return
    current_year = latest_year_from_df(source, year_column=year_column)
    if current_year is None:
        print(f"Anno mancante per {title}")
        return
    latest = source[source[year_column] == current_year]
    top_codes = latest.sort_values(value_column, ascending=False).head(top)[id_column].tolist()
    selected = source[source[id_column].isin(top_codes)]
    matrix = selected.pivot_table(index=year_column, columns=label_column, values="value", aggfunc="first").sort_index()
    if matrix.empty:
        print(f"Nessuna serie per {title}")
        return
    matrix.plot(figsize=(12, 6), marker="o")
    plt.title(title)
    plt.ylabel(value_column)
    plt.xlabel("Anno")
    plt.tight_layout()
    plt.show()


def summarize_section(payload):
    print("Chiavi sezione:", list(payload.keys()))
    for root_key, root_value in payload.items():
        if isinstance(root_value, list):
            print(f"{root_key}: list ({len(root_value)})")
        elif isinstance(root_value, dict):
            print(f"{root_key}: dict")
            for sub_key, sub_value in root_value.items():
                if isinstance(sub_value, list):
                    print(f"  - {root_key}.{sub_key}: {len(sub_value)} righe")
                elif isinstance(sub_value, dict):
                    print(f"  - {root_key}.{sub_key}: dict")
                else:
                    print(f"  - {root_key}.{sub_key}: {type(sub_value).__name__}")
        elif root_value is not None:
            print(f"{root_key}: {type(root_value).__name__}")


## Elenco opzioni disponibili

Esegui questa cella prima delle celle dove imposti parametri:

- controlla i nomi ammessi
- usa esattamente i valori indicati
- verifica i codici/regioni/anni disponibili nella tua versione dati


In [ ]:
# Opzioni Italia - valori ammessi

def print_values(header, values):
    print(header)
    if not values:
        print("  (nessun valore)")
        return
    for item in values:
        print(f" - {item}")

revenue_categories = rows_to_df(section.get("revenue", {}).get("category_series", []))
spending_categories = rows_to_df(section.get("spending", {}).get("category_series", []))
revenue_top = rows_to_df(section.get("revenue", {}).get("top_taxes", []))
spending_by_function = rows_to_df(section.get("spending", {}).get("by_function", []))
pressure_trend = rows_to_df(section.get("revenue", {}).get("pressure_trend", []))

metric_candidates = [
    "latest_percent_pil",
    "latest_value_mld",
    "value",
    "value_mld",
    "value_pil",
    "value_pil_percent",
    "share_of_total",
    "share_spesa_totale",
    "latest_year",
]

VALUE_PREFERENCE_OPTIONS = sorted(
    {
        field
        for field in metric_candidates
        if field in revenue_categories.columns
        or field in revenue_top.columns
        or field in spending_categories.columns
        or field in spending_by_function.columns
    }
)

TOP_N_MIN = 3
TOP_N_MAX = 100

year_options = {
    "tax_pressure": section.get("latest_years", {}).get("tax_pressure"),
    "spending": section.get("latest_years", {}).get("spending"),
    "revenue_items": section.get("latest_years", {}).get("revenue_items"),
}

revenue_labels = []
spending_labels = []
if not revenue_categories.empty and "label" in revenue_categories.columns:
    revenue_labels = sorted(set(revenue_categories["label"].dropna().astype(str).tolist()))
if not spending_categories.empty and "label" in spending_categories.columns:
    spending_labels = sorted(set(spending_categories["label"].dropna().astype(str).tolist()))

pressure_years = []
if not pressure_trend.empty and "year" in pressure_trend.columns:
    pressure_years = sorted(set(pd.to_numeric(pressure_trend["year"], errors="coerce").dropna().astype(int).tolist()))

print("=== Parametri disponibili (Italia) ===")
print(f"Anni sintetici disponibili: {year_options}")
print(f"Metriche ammesse per VALUE_PREFERENCE: {VALUE_PREFERENCE_OPTIONS}")
print(f"Intervallo consigliato TOP_N: {TOP_N_MIN}..{TOP_N_MAX}")
print(f"Anni disponibili per trend pressione: {pressure_years}")

print()
print("Etichette categorie entrate (valori possibili per filtrare/ordinare):")
print_values("Totale voci: %s" % len(revenue_labels), revenue_labels)

print()
print("Etichette categorie spesa:")
print_values("Totale voci: %s" % len(spending_labels), spending_labels)

print()
print("Nota: questo notebook lavora sul riepilogo Italia; per le regioni vai su 04_regioni.ipynb")

print()
print("Esempio di codice da copiare:")
print("TOP_N = 10")
print("VALUE_PREFERENCE = ['latest_percent_pil', 'latest_value_mld', 'value', 'value_mld']")
print("TOP_N = 20  # se vuoi più righe")

ITALIA_VALUE_OPTIONS = VALUE_PREFERENCE_OPTIONS
ITALIA_TOP_N_RANGE = [TOP_N_MIN, TOP_N_MAX]
ITALIA_YEAR_OPTIONS = pressure_years
ITALIA_REVENUE_LABELS = revenue_labels
ITALIA_SPENDING_LABELS = spending_labels


## Stato sezione

Caricamento completo di blocchi, righe e schemi.

In [ ]:
kpis = rows_to_df(section.get("kpis", []))
revenue = section.get("revenue", {})
spending = section.get("spending", {})
distribution = section.get("distribution", {})

display(kpis)
print("Anni sintetici:", section.get("latest_years", {}))
print("Fonti:")
print(section.get("source_updates", {}))
print("Percorso source payload:", status.get("source_json"))

summarize_section(section)


## Pressione fiscale e componenti

Totale fiscale, composizione e andamento nel tempo.

In [ ]:
pressure = rows_to_df(revenue.get("pressure_trend", []))
if pressure.empty:
    print("pressure_trend assente")
else:
    pressure[["year", "value"]].dropna().sort_values("year").plot(
        x="year", y="value", marker="o", figsize=(12, 4)
    )
    plt.title("Pressione fiscale e contributiva - totale")
    plt.ylabel("% PIL")
    plt.tight_layout()
    plt.show()

component_candidates = [
    "imposte su produzione e importazioni",
    "imposte su reddito",
    "imposte su patrimonio",
    "contributi sociali netti",
    "imposte in conto capitale",
]
component_cols = [item for item in component_candidates if item in pressure.columns]
if component_cols:
    pressure[["year"] + component_cols].sort_values("year").set_index("year").plot(figsize=(12, 6))
    plt.title("Composizione pressione fiscale")
    plt.ylabel("% PIL")
    plt.tight_layout()
    plt.show()

pressure_components = rows_to_df(revenue.get("pressure_components", []))
if not pressure_components.empty:
    latest_year = latest_year_from_df(pressure_components)
    if latest_year:
        pressure_components[pressure_components["year"] == latest_year].sort_values("value", ascending=False).set_index("year").plot(kind="bar", figsize=(10, 4))
        plt.title(f"Composizione pressione fiscale {latest_year}")
        plt.tight_layout()
        plt.show()


## Entrate

Confronti a valore, quota PIL e serie storiche per categoria.

In [ ]:
category_rows = rows_to_df(revenue.get("category_series", []))
if category_rows.empty:
    print("Nessuna categoria entrate")
else:
    bar_field = pick_metric(category_rows.iloc[0].to_dict(), ["latest_value_mld", "latest_percent_pil", "value"])
    if bar_field:
        top_bars(category_rows, bar_field, "Composizione entrate ultimi valori", top=12)
    series_mld = explode_series(revenue.get("category_series", []), "value_mld")
    if not series_mld.empty:
        plot_series(series_mld, "value_mld", "Serie storiche categoria entrate", top=8)
        series_pct = series_mld.dropna(subset=["percent_pil"]).copy()
        if not series_pct.empty:
            series_pct = series_pct.rename(columns={"percent_pil": "value"})
            plot_series(series_pct, "value", "Serie storiche categoria entrate in % PIL", top=8)

all_lines = rows_to_df(revenue.get("all_lines", []))
if not all_lines.empty:
    metric_line = pick_metric(all_lines.iloc[0].to_dict(), ["latest_value_mld", "latest_percent_pil", "value"])
    if metric_line:
        top_bars(all_lines, metric_line, "Principali categorie entrate", top=15)

top_taxes = rows_to_df(revenue.get("top_taxes", []))
if not top_taxes.empty:
    top_taxes_value = pick_metric(top_taxes.iloc[0].to_dict(), ["latest_value_mld", "latest_percent_pil", "value"])
    if top_taxes_value:
        top_bars(top_taxes, top_taxes_value, "Top entrate principali", top=8)

direct_indirect = rows_to_df(revenue.get("direct_indirect", []))
if not direct_indirect.empty:
    if {"Dirette", "Indirette"}.issubset(set(direct_indirect.columns)):
        direct_indirect[["year", "Dirette", "Indirette"]].sort_values("year").set_index("year").plot(figsize=(10, 4), marker="o")
        plt.title("Entrate dirette e indirette")
        plt.tight_layout()
        plt.show()


## Spesa

Funzioni COFOG, focus e dettagli per codice/anno.

In [ ]:
focus_rows = rows_to_df(spending.get("focus", []))
if not focus_rows.empty:
    top_bars(focus_rows, "value_mld", "Focus principale spesa", top=12)

spend_categories = rows_to_df(spending.get("category_series", []))
if not spend_categories.empty:
    top_bars(spend_categories, "latest_value_mld", "Spesa per funzione COFOG - ultimi valori", top=12)
    series = explode_series(spending.get("category_series", []), "value_mld")
    if not series.empty:
        plot_series(series, "value", "Serie storiche spesa per funzione", top=10)

by_function = rows_to_df(spending.get("by_function", []))
if not by_function.empty:
    by_function[["year", "value_mld"]].dropna().set_index("year").plot(figsize=(10, 4), marker="o")
    plt.title("Totale spesa (mld)")
    plt.tight_layout()
    plt.show()

detail_rows = explode_series(spending.get("function_detail_series", []), "value_mld")
if detail_rows.empty:
    print("Dettaglio COFOG non disponibile")
else:
    parent_filter = None
    if parent_filter:
        to_plot = detail_rows[detail_rows["code"].str.startswith(parent_filter)]
    else:
        to_plot = detail_rows
    plot_series(to_plot, "value_mld", "Dettaglio COFOG nel tempo", top=12)


## Distribuzione e dichiarazioni

Fasce IRPEF, distribuzione redditi, patrimonio e successioni.

In [ ]:
irpef_bands = rows_to_df(distribution.get("irpef_by_band", []))
irpef_share = rows_to_df(distribution.get("irpef_share_by_band", []))
income_share = rows_to_df(distribution.get("income_distribution_by_band", []))
wealth = rows_to_df(distribution.get("household_wealth", []))
if wealth.empty:
    wealth = rows_to_df(distribution.get("household_wealth_distribution", {}))

if not irpef_bands.empty:
    irpef_bands.plot(x="band", y="contributors", kind="bar", figsize=(12, 4), legend=False)
    plt.title("Contribuenti IRPEF per banda")
    plt.tight_layout()
    plt.show()
if not irpef_share.empty:
    irpef_share.plot(x="band", y="tax_share", kind="bar", figsize=(12, 4), legend=False)
    plt.title("Quota di gettito IRPEF")
    plt.tight_layout()
    plt.show()
if not income_share.empty:
    income_share[["band", "work", "capital_gain", "capital_income", "total_mld"]].set_index("band").plot(kind="bar", figsize=(13, 5))
    plt.title("Distribuzione redditi per banda")
    plt.tight_layout()
    plt.show()

print("\nDichiarazioni")
display(rows_to_df(distribution.get("declarations", {})))

succession = rows_to_df([distribution.get("succession_gift_tax", {})]) if isinstance(distribution.get("succession_gift_tax", {}), dict) else rows_to_df(distribution.get("succession_gift_tax", []))
display(succession)
if not succession.empty:
    succession_series = rows_to_df(succession.iloc[0].get("series") if not succession.empty and isinstance(succession.iloc[0], dict) else [])
    if not succession_series.empty:
        succession_series = succession_series.rename(columns={"year": "anno", "value_million_euro": "valore"})
        succession_series.plot(x="anno", y="valore", marker="o", figsize=(10, 3))
        plt.title("Successioni e donazioni nel tempo")
        plt.tight_layout()
        plt.show()

if not wealth.empty:
    display(wealth)


## Laboratorio grafici (molti output)

Regola i parametri per creare rapidamente tante viste da serie disponibili.

In [ ]:
TOP_N = 10

# Scegli se vuoi usare i valori in miliardi o in quota PIL
VALUE_PREFERENCE = ["latest_percent_pil", "latest_value_mld", "value", "value_mld"]

series_candidates = {
    "Entrate categorie": rows_to_df(revenue.get("category_series", [])),
    "Spesa categorie": rows_to_df(spending.get("category_series", [])),
    "Spesa focus": rows_to_df(spending.get("focus", [])),
}

for title, rows in series_candidates.items():
    if rows.empty:
        continue
    metric = pick_metric(rows.iloc[0].to_dict(), VALUE_PREFERENCE)
    if metric:
        top_bars(rows, metric, f"{title} - {metric}", top=TOP_N)
    if "series" in rows.iloc[0].to_dict() or any("series" in row for row in rows.to_dict("records")):
        exploded = explode_series(rows.to_dict("records"), "value_mld")
        if not exploded.empty:
            plot_series(exploded, "value", f"Serie storiche {title}", top=TOP_N)
